# Конструювання підказок | Prompt Engineering

In [ ]:
import os
import openai
from dotenv import load_dotenv

load_dotenv()

key = os.getenv('OPEN_API_KEY')
client = openai.OpenAI(api_key=key)

# helper function
def get_completion(prompt, model="gpt-4o"):
    response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7  # Adjust creativity
        )
    return response.choices[0].message.content

**Basics of prompt engineering.**

In [10]:
text = f"""
wait client.connect();
    const database = client.db("your-database-name");
    const collection = database.collection("your-collection-name");

    // Example: Insert a document
    const result = await collection.insertOne();

    // Example: Retrieve documents
    const documents = await collection.find().toArray();
"""
prompt = f"""
Summarize the code delimited by triple backticks \ 
into a single very small sentence.
```{text}```
"""
response = get_completion(prompt)
print(response)

The code connects to a database, inserts a document, and retrieves documents from a collection.


**Інструкція**

Ви можете створювати ефективні підказки для різних простих завдань, використовуючи команди, щоб вказати моделі, чого ви хочете досягти, наприклад «Написати», «Класифікувати», «Підсумувати», «Перекласти», «Впорядкувати» тощо.

Майте на увазі, що вам також потрібно багато експериментувати, щоб побачити, що працює найкраще. Спробуйте різні інструкції з різними ключовими словами, контекстами та даними і подивіться, що найкраще підходить для вашого конкретного випадку використання та завдання. Зазвичай, чим конкретнішим і релевантнішим є контекст для завдання, яке ви намагаєтеся виконати, тим краще. Ми торкнемося важливості вибірки та додавання більше контексту в наступних посібниках.

Інші рекомендують розміщувати інструкції на початку підказки. Ще одна рекомендація — використовувати чіткий роздільник, наприклад «###», щоб відокремити інструкцію від контексту.

In [31]:
prompt = f"""
### Instruction ###
Translate the text below to Spanish:
Text: "hello!"
"""
# Tone translation

prompt = f"""
### Instruction ###
Translate the following from slang to a business letter: 
Text: 'Dude, This is Joe, check out this spec on this standing lamp.'
"""
response = get_completion(prompt)
print(response)

Subject: Review of Standing Lamp Specifications

Dear [Recipient's Name],

I hope this message finds you well. My name is Joe, and I would like to bring to your attention the specifications of a standing lamp that may be of interest to you. 

Please take a moment to review the attached document detailing the lamp's features and specifications.

Thank you for your time and consideration.

Best regards,

Joe


**Zero-Shot Prompting**

Сучасні великі мовні моделі (LLM), такі як GPT-3.5 Turbo, GPT-4 і Claude 3, налаштовані на виконання інструкцій і навчені на великих обсягах даних. Завдяки масштабному навчанню ці моделі здатні виконувати деякі завдання в режимі «zero-shot». Zero-shot prompting означає, що підказка, яка використовується для взаємодії з моделлю, не міститиме прикладів або демонстрацій. Zero-shot prompt безпосередньо дає моделі вказівку виконати завдання без додаткових прикладів для її керування.

In [ ]:
prompt = f"""
Classify the text into neutral, negative or positive. 
Text: `I think the vacation is okay.`
Sentiment:
"""


**Few-Shot Prompting**

In [25]:
prompt = f"""
A "whatpu" is a small, furry animal native to Tanzania. An example of a sentence that uses the word whatpu is:
We were traveling in Africa and we saw these very cute whatpus.
 
To do a "farduddle" means to jump up and down really fast. An example of a sentence that uses the word farduddle is:
"""

response = get_completion(prompt)
print(response)

During the celebration, the children couldn't contain their excitement and began to farduddle in the courtyard.


**Chain-of-Thought Prompting**

Introduced in Wei et al. (2022), chain-of-thought (CoT) prompting enables complex reasoning capabilities through intermediate reasoning steps. You can combine it with few-shot prompting to get better results on more complex tasks that require reasoning before responding.

In [ ]:
# Zero-shot COT Prompting - adding "Let's think step by step" to the original prompt
prompt = "Скільки букв `р` в слові `Абракадабра`?"
# prompt += " Давай думати крок за кроком."

response = get_completion(prompt)
print(response)

# Proper Chain-of-Thought Prompting
# prompt = """
# Скільки букв `д` в слові `Абракадабра`?
# Відповідь: одна `д` на 7-й позиції
# Скільки букв `б` в слові `Абракадабра?
# Відповідь: дві `б` на 2-й та на 9-й позиціях
# Скільки букв `р` в слові `Абракадабра`?"""
# response = get_completion(prompt)
# print(response)

В слові "Абракадабра" дві літери `р`, на 3-й та 10-й позиціях.


**Дерево Думок (Tree of Thoughts, ToT)**

ToT підтримує дерево думок, де думки представляють собою послідовні мовні послідовності, які служать проміжними кроками на шляху до вирішення проблеми. Такий підхід дозволяє LM самостійно оцінювати прогрес, досягнутий за допомогою проміжних думок, на шляху до вирішення проблеми через обдуманий процес міркування. Здатність LM генерувати та оцінювати думки потім поєднується з алгоритмами пошуку (наприклад, пошук в ширину та пошук в глибину), щоб забезпечити систематичне дослідження думок з попереднім переглядом та поверненням назад.

In [33]:
# question = "Скільки букв `р` в слові `Абракадабра`?"
question = "The bottom of the mug was cut off and the top was sealed. Can you drink water from it?"
# question = "I have to wash my car at the carwash. It is 50 meters away from my home. Should I drive there, or get there by walking?"
# response = get_completion(question)
# print(response)

prompt = f""" Imagine three different experts are answering this question.
All experts will write down 1 step of their thinking,
then share it with the group.
Then all experts will go on to the next step, etc.
If any expert realises they're wrong at any point then they leave.
The question is: ```{question}```"""
response = get_completion(prompt)
print(response)

**Expert 1:**

Step 1: Consider the physical structure of the mug once the bottom is cut off. Since the bottom is removed, the mug effectively becomes an open cylinder.

**Expert 2:**

Step 1: Analyze the changes to the mug—by having both the top of the mug sealed and the bottom cut off, the transformation must significantly compromise its capacity to hold liquid.

**Expert 3:**

Step 1: Think about the mug's purpose. Typically, a mug must hold water without leaking for someone to drink properly from it.

---

**Expert 1:**

Step 2: Even with the top sealed, the lack of a bottom section means there's no enclosure to store water until commitment to drink happens.

**Expert 2:**

Step 2: Since gravity will cause the liquid to immediately fall through due to the missing bottom, the mug fails its storage state—a requirement to delivering sips of droplet content directly... naturally (fr================ way previously rely(

**Expert 3:** notes_sb-O_ai)

Restart verbosity; higher end smell-